In [22]:
import pandas as pd
import numpy as np
import torch
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import os

In [23]:
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])
model.to(device)
model.eval()

torch.backends.cudnn.benchmark = True

In [24]:
preprocess = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [25]:
session = requests.Session()

def load_image(url):

    try:
        r = session.get(url, timeout=7)

        if r.status_code != 200:
            return None

        img = Image.open(BytesIO(r.content)).convert("RGB")
        img = preprocess(img)

        return img

    except:
        return None

In [26]:
def load_images_parallel(urls, ids, workers=32):

    images = []
    valid_ids = []

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(load_image, urls))

    for img, pid in zip(results, ids):
        if img is not None:
            images.append(img)
            valid_ids.append(pid)

    return images, valid_ids

In [27]:
csv_path = "usa_europe_geotagged.csv"

df = pd.read_csv(csv_path, nrows=20000)

df.head()

,photoid,uid,unickname,datetaken,dateuploaded,capturedevice,title,description,usertags,machinetags,...,pageurl,downloadurl,licensename,licenseurl,serverid,farmid,secret,secretoriginal,ext,marker
0,29872,34427465504@N01,Trinity,2004-05-05 21:30:05.0,1083817805,NaN,bruise+5%2F2%2F4,I+got+this+bruise+at+work...wonder+if+I+can+ge...,"2004,bruise,me,photo,unfound",NaN,...,http://www.flickr.com/photos/34427465504@N01/2...,http://farm1.staticflickr.com/1/29872_c787b1b0...,Attribution-ShareAlike License,http://creativecommons.org/licenses/by-sa/2.0/,1,1,c787b1b056,c787b1b056,jpg,0
1,29873,34427465504@N01,Trinity,2004-05-05 21:30:06.0,1083817806,NaN,bruise+5%2F5%2F4,same+bruise%2C+day+4,"2004,bruise,me,photo,unfound",NaN,...,http://www.flickr.com/photos/34427465504@N01/2...,http://farm1.staticflickr.com/1/29873_d0359535...,Attribution-ShareAlike License,http://creativecommons.org/licenses/by-sa/2.0/,1,1,d0359535ab,d0359535ab,jpg,0
2,29874,34427465504@N01,Trinity,2004-05-05 21:30:07.0,1083817807,NaN,bruise+5%2F3%2F4,same+bruise%2C+day+2,"2004,bruise,me,photo,unfound",NaN,...,http://www.flickr.com/photos/34427465504@N01/2...,http://farm1.staticflickr.com/1/29874_f02c63c2...,Attribution-ShareAlike License,http://creativecommons.org/licenses/by-sa/2.0/,1,1,f02c63c2c8,f02c63c2c8,jpg,0
3,29875,34427465504@N01,Trinity,2004-05-05 21:30:09.0,1083817809,NaN,bruise+5%2F4%2F4,same+bruise%2C+day+3,"2004,bruise,me,photo,unfound",NaN,...,http://www.flickr.com/photos/34427465504@N01/2...,http://farm1.staticflickr.com/1/29875_6251beed...,Attribution-ShareAlike License,http://creativecommons.org/licenses/by-sa/2.0/,1,1,6251beed11,6251beed11,jpg,0
4,30431,34427469121@N01,George,2004-03-05 15:03:52.0,1083960838,FUJIFILM+FinePix+S5000,Space+age,This+is+the+little+lift+which+goes+up+the+Spac...,"elevator,seattle,space+needle,space+travel",NaN,...,http://www.flickr.com/photos/34427469121@N01/3...,http://farm1.staticflickr.com/1/30431_37ed41d3...,Attribution-NonCommercial-NoDerivs License,http://creativecommons.org/licenses/by-nc-nd/2.0/,1,1,37ed41d3fb,37ed41d3fb,jpg,0


In [28]:
df.columns

Index(['photoid', 'uid', 'unickname', 'datetaken', 'dateuploaded',
       'capturedevice', 'title', 'description', 'usertags', 'machinetags',
       'longitude', 'latitude', 'accuracy', 'pageurl', 'downloadurl',
       'licensename', 'licenseurl', 'serverid', 'farmid', 'secret',
       'secretoriginal', 'ext', 'marker'],
      dtype='object')

In [35]:
from tqdm import tqdm

def generate_embeddings(urls, ids, chunk_id, batch_size=256):

    embeddings = []
    valid_ids = []

    total_batches = (len(urls) + batch_size - 1) // batch_size

    for start in tqdm(
        range(0, len(urls), batch_size),
        total=total_batches,
        desc=f"Chunk {chunk_id}",
        unit="batch"
    ):

        batch_urls = urls[start:start+batch_size]
        batch_ids = ids[start:start+batch_size]

        images, good_ids = load_images_parallel(batch_urls, batch_ids)

        if len(images) == 0:
            continue

        batch = torch.stack(images).to(device)

        with torch.no_grad():
            emb = model(batch)

        emb = emb.view(emb.size(0), -1).cpu().numpy()

        embeddings.append(emb)
        valid_ids.extend(good_ids)

    if len(embeddings) == 0:
        return None, None

    embeddings = np.concatenate(embeddings, axis=0)

    return embeddings, np.array(valid_ids)

In [36]:
csv_path = "usa_europe_geotagged.csv"

reader = pd.read_csv(csv_path, chunksize=10000)

In [37]:
import pandas as pd
import numpy as np
import os

reader = pd.read_csv("usa_europe_geotagged.csv", chunksize=10000)

EMB_DIR = "embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

START_CHUNK = 693

for i, chunk in enumerate(reader):

    if i < START_CHUNK:
        continue

    urls = chunk["downloadurl"].tolist()
    ids = chunk["photoid"].tolist()

    embeddings, valid_ids = generate_embeddings(urls, ids, i)

    if embeddings is None:
        print(f"Chunk {i}: no valid images")
        continue

    np.save(f"{EMB_DIR}/embeddings_chunk_{i}.npy", embeddings)
    np.save(f"{EMB_DIR}/ids_chunk_{i}.npy", valid_ids)

Chunk 767: 100%|██████████| 10/10 [00:45<00:00,  4.53s/batch]
